# Trustworthy Pneumonia Detection — Google Colab Demo

Use this notebook to train and show the project to faculty.

**Before anything else:** `Runtime` → `Change runtime type` → **T4 GPU** → Save.

Pipeline: dataset → train ResNet18 → temperature scaling → evaluation → Grad-CAM + TrustScore on one X-ray.

## 1. Get the project onto Colab

Preferred: upload a zip of **this local repo** (it includes the bugfixes).  
Fallback: clone GitHub if that copy is up to date.

In [ ]:
from pathlib import Path
import os, zipfile, shutil

PROJECT_NAME = "Trustworthy-Pneumonia-Detection-XAI"
ROOT = Path("/content") / PROJECT_NAME

# Option A — upload the project zip in the Colab file sidebar, then set this path.
ZIP_PATH = Path("/content/Trustworthy-Pneumonia-Detection-XAI.zip")

# Option B — clone (only if your GitHub copy has the latest fixes)
CLONE_URL = ""  # e.g. "https://github.com/HARIKRISHNA39/Trustworthy-Pneumonia-Detection-XAI.git"

if ROOT.exists():
    print("Project already at", ROOT)
elif ZIP_PATH.exists():
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall("/content")
    extracted = [p for p in Path("/content").iterdir() if p.is_dir() and PROJECT_NAME.lower() in p.name.lower()]
    if extracted and extracted[0] != ROOT:
        shutil.move(str(extracted[0]), str(ROOT))
    print("Unzipped to", ROOT)
elif CLONE_URL:
    !git clone {CLONE_URL} {ROOT}
else:
    raise FileNotFoundError(
        "Upload a zip of the project to /content, or set CLONE_URL. "
        "In Colab: left sidebar → Files → Upload."
    )

os.chdir(ROOT)
print("Working directory:", Path.cwd())

In [ ]:
!pip -q install -r requirements.txt kaggle
import torch
print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("GPU is off. Runtime → Change runtime type → T4 GPU, then rerun from the top.")

## 2. Download the Kaggle dataset

1. Open [Kaggle account settings](https://www.kaggle.com/settings) → **Create New Token** (`kaggle.json`).
2. Run the next cell and upload that file.
3. Accept the dataset license once in the browser: [chest-xray-pneumonia](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia).

In [ ]:
from google.colab import files
from pathlib import Path

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
token = kaggle_dir / "kaggle.json"

if not token.exists():
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise FileNotFoundError("Upload kaggle.json")
    token.write_bytes(uploaded["kaggle.json"])
token.chmod(0o600)

zip_name = "chest-xray-pneumonia.zip"
if not Path(zip_name).exists() and not Path("/content/chest_xray").exists():
    !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
    !unzip -q -o {zip_name} -d /content

candidates = [
    Path("/content/chest_xray"),
    Path("/content/chest_xray/chest_xray"),
]
kaggle_root = next((p for p in candidates if (p / "train").exists()), None)
if kaggle_root is None:
    raise FileNotFoundError("Could not find the unzipped chest_xray/train folder")
print("Kaggle data:", kaggle_root)

In [ ]:
!python scripts/prepare_dataset.py "{kaggle_root}" --val_ratio 0.15
from datasets.dataset import ChestXRayDataset
from configs.config import DATA_DIR

for split in ("train", "val", "test"):
    n = len(ChestXRayDataset(root=str(DATA_DIR / split)))
    print(f"{split}: {n} images")

## 3. Train, calibrate, evaluate

- `QUICK = True` → about 2 epochs on 128 images/split. Good for a live faculty demo.
- `QUICK = False` → 15 epochs on the full set. Use this if you have 30–60 minutes on GPU.

In [ ]:
QUICK = True  # set False for the full research run

if QUICK:
    !python -m training.train --model_type resnet18 --quick
else:
    !python -m training.train --model_type resnet18 --epochs 15

!python -m evaluation.calibrate
!python -m evaluation.evaluate

## 4. Upload an X-ray and predict

Run the next cell, click **Choose Files**, and pick a `.jpg` / `.jpeg` / `.png` chest X-ray.

You must run the **training cell first**. Without `checkpoints/chestxray_best.pt` prediction cannot start.

In [ ]:
from pathlib import Path
from google.colab import files
from IPython.display import Image, display

from configs.config import DATA_DIR, MODEL_SAVE_DIR, MODEL_WEIGHTS_NAME, OUTPUT_DIR, ensure_dirs
from datasets.dataset import ChestXRayDataset
from inference.api import predict

ensure_dirs()
ckpt = MODEL_SAVE_DIR / MODEL_WEIGHTS_NAME
if not ckpt.exists():
    raise FileNotFoundError(
        f"No trained model at {ckpt}. Scroll up and run the training cell first."
    )

print("Click Choose Files and upload a chest X-ray (.jpg / .jpeg / .png)")
uploaded = files.upload()

if uploaded:
    upload_dir = Path("uploads")
    upload_dir.mkdir(exist_ok=True)
    name = next(iter(uploaded))
    image_path = upload_dir / Path(name).name
    image_path.write_bytes(uploaded[name])
else:
    test_ds = ChestXRayDataset(root=str(DATA_DIR / "test"))
    if len(test_ds) == 0:
        raise FileNotFoundError("No image uploaded and no test images found.")
    image_path = Path(test_ds.samples[0][0])
    print("No upload detected. Using a test-set image instead:", image_path)

print("Predicting:", image_path)
result = predict(str(image_path))
print("\nRESULT")
for key in (
    "predicted_class",
    "raw_confidence",
    "calibrated_confidence",
    "uncertainty_score",
    "focus_score",
    "trust_score",
    "trust_level",
    "ood_flag",
):
    print(f"{key:24} {result[key]}")

fig_path = result.get("overlay_path")
if fig_path and Path(fig_path).exists():
    display(Image(fig_path, width=900))
else:
    print("Prediction finished but no figure was saved.")

In [ ]:
from IPython.display import Image, display
from pathlib import Path

outputs = Path("outputs")
for name in [
    "training_curves.png",
    "reliability_diagrams.png",
    "evaluation_results.png",
]:
    path = outputs / name
    if path.exists():
        print(path)
        display(Image(str(path), width=720))

for path in sorted(outputs.glob("*_uncertainty_gradcam.png")):
    print(path)
    display(Image(str(path), width=900))

## 5. Faculty figure pack (run this last)

Generates **12 supporting graphs** for the presentation: metrics, ROC, calibration, TrustScore, uncertainty, FocusScore, and a Grad-CAM gallery.

Takes a few minutes on GPU. Then every figure is shown below.

In [ ]:
!python -m evaluation.faculty_report --max_samples 160 --n_trust 48 --n_mc 12 --n_gradcam 8

from IPython.display import Image, display
from pathlib import Path

report = Path("outputs/faculty_report")
figures = sorted(report.glob("*.png"))
if not figures:
    print("No figures yet. Run training first, then re-run this cell.")
else:
    for path in figures:
        print("\n" + path.name)
        display(Image(str(path), width=820))

## 6. Streamlit frontend (optional, for faculty)

Launches the same pipeline in a browser UI: upload an X-ray, see metrics + Grad-CAM, and browse the graphs.

A public Cloudflare link is printed. Open it on the projector. Keep this cell running.

In [ ]:
!pip -q install streamlit
!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

import subprocess, time, re, os
from pathlib import Path

os.chdir("/content/Trustworthy-Pneumonia-Detection-XAI")
assert Path("app.py").exists(), "Re-upload the latest project zip so app.py is included."
assert Path("checkpoints/chestxray_best.pt").exists(), "Run the training cell first."

# Restart-safe: kill an older Streamlit if you re-run this cell
subprocess.call(["pkill", "-f", "streamlit run app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

app = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true",
     "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
time.sleep(8)

tunnel = subprocess.Popen(
    ["/tmp/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url = None
for _ in range(40):
    line = tunnel.stdout.readline() if tunnel.stdout else ""
    print(line, end="")
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line or "")
    if match:
        url = match.group(0)
        break
    time.sleep(0.4)

if url:
    print("\nOpen this link for the faculty UI:\n", url)
else:
    print("Tunnel URL not found. Scroll the cloudflared output for a trycloudflare.com link.")

## Faculty talking points

1. Prediction is only the first output.
2. Temperature scaling makes confidence honest (show the reliability diagram).
3. Grad-CAM shows whether the model looked at lungs, not text/edges.
4. TrustScore + OOD flag is the safety layer.

This is a **research prototype** on a public Kaggle dataset, not a clinical device.

Download `outputs/` from the Colab file sidebar if you need PNGs in a slide deck.